# Decision Stump

A Decision Stump is the simplest form of a decision tree - it has only one internal node (the root) which is immediately connected to the terminal nodes (leaves). It makes a decision based on the value of a single feature.

## Key Concepts:

- **Single Split**: Makes only one decision based on one feature
- **Weak Learner**: Often used as a weak learner in ensemble methods like AdaBoost
- **Simple Interpretation**: Highly interpretable and explainable
- **Fast Training**: Very fast to train due to simplicity
- **Baseline Model**: Often used as a baseline for comparison

## When to Use:

- As a weak learner in boosting algorithms
- When you need a simple, interpretable baseline model
- When computational resources are very limited
- When features have strong individual predictive power
- For quick prototyping and feature importance analysis

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris, make_classification
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.ensemble import AdaBoostClassifier

# Set style for better visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

## Decision Stump Implementation

A decision stump is essentially a decision tree with max_depth=1.

In [ ]:
class DecisionStump:
    """Decision Stump Classifier - Simple one-level decision tree"""
    
    def __init__(self):
        self.feature_idx = None
        self.threshold = None
        self.left_class = None
        self.right_class = None
        self.feature_name = None
    
    def fit(self, X, y, feature_names=None):
        """Find the best single split"""
        n_samples, n_features = X.shape
        best_gini = float('inf')
        
        if feature_names is None:
            feature_names = [f'Feature_{i}' for i in range(n_features)]
        
        # Try each feature
        for feature_idx in range(n_features):
            feature_values = X[:, feature_idx]
            unique_values = np.unique(feature_values)
            
            # Try each possible threshold
            for threshold in unique_values:
                # Split
                left_mask = feature_values <= threshold
                right_mask = feature_values > threshold
                
                # Skip if split is invalid
                if np.sum(left_mask) == 0 or np.sum(right_mask) == 0:
                    continue
                
                # Calculate Gini impurity
                gini = self._calculate_gini(y, left_mask, right_mask)
                
                # Update if better
                if gini < best_gini:
                    best_gini = gini
                    self.feature_idx = feature_idx
                    self.threshold = threshold
                    self.feature_name = feature_names[feature_idx]
                    
                    # Determine class for each side
                    self.left_class = self._majority_class(y[left_mask])
                    self.right_class = self._majority_class(y[right_mask])
    
    def _calculate_gini(self, y, left_mask, right_mask):
        """Calculate weighted Gini impurity"""
        n = len(y)
        n_left = np.sum(left_mask)
        n_right = np.sum(right_mask)
        
        gini_left = self._gini_impurity(y[left_mask])
        gini_right = self._gini_impurity(y[right_mask])
        
        weighted_gini = (n_left / n) * gini_left + (n_right / n) * gini_right
        return weighted_gini
    
    def _gini_impurity(self, y):
        """Calculate Gini impurity for a set of labels"""
        if len(y) == 0:
            return 0
        
        unique_labels, counts = np.unique(y, return_counts=True)
        probabilities = counts / len(y)
        gini = 1 - np.sum(probabilities ** 2)
        return gini
    
    def _majority_class(self, y):
        """Return the majority class"""
        if len(y) == 0:
            return 0
        unique_labels, counts = np.unique(y, return_counts=True)
        return unique_labels[np.argmax(counts)]
    
    def predict(self, X):
        """Make predictions"""
        predictions = []
        for sample in X:
            if sample[self.feature_idx] <= self.threshold:
                predictions.append(self.left_class)
            else:
                predictions.append(self.right_class)
        return np.array(predictions)
    
    def __repr__(self):
        return (f"DecisionStump(feature='{self.feature_name}', "
                f"threshold={self.threshold:.4f}, "
                f"left_class={self.left_class}, "
                f"right_class={self.right_class})")

## Dataset 1: Iris Dataset

Let's test the decision stump on the Iris dataset.

In [ ]:
# Load Iris dataset
iris = load_iris()
X = iris.data
y = iris.target
feature_names = iris.feature_names
target_names = iris.target_names

print(f"Feature shape: {X.shape}")
print(f"Classes: {target_names}")
print(f"Features: {feature_names}")
print(f"\nClass distribution: {np.bincount(y)}")

In [ ]:
# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"Training samples: {X_train.shape[0]}")
print(f"Test samples: {X_test.shape[0]}")

## Train Custom Decision Stump

In [ ]:
# Train custom decision stump
stump = DecisionStump()
stump.fit(X_train, y_train, feature_names)

print("Decision Stump:")
print(stump)

# Make predictions
y_pred = stump.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"\nAccuracy: {accuracy:.4f}")

## Using sklearn's DecisionTreeClassifier as Decision Stump

We can also use sklearn's DecisionTreeClassifier with max_depth=1.

In [ ]:
# Train sklearn decision stump
stump_sklearn = DecisionTreeClassifier(
    max_depth=1,  # This makes it a decision stump
    criterion='gini',
    random_state=42
)

stump_sklearn.fit(X_train, y_train)
y_pred_sklearn = stump_sklearn.predict(X_test)
accuracy_sklearn = accuracy_score(y_test, y_pred_sklearn)

print(f"Sklearn Decision Stump Accuracy: {accuracy_sklearn:.4f}")
print(f"Tree depth: {stump_sklearn.get_depth()}")
print(f"Number of leaves: {stump_sklearn.get_n_leaves()}")

## Visualize Decision Stump

In [ ]:
# Plot the decision stump
plt.figure(figsize=(12, 8))
plot_tree(stump_sklearn, 
          feature_names=feature_names,
          class_names=target_names,
          filled=True,
          rounded=True,
          fontsize=12)
plt.title('Decision Stump (max_depth=1) - Iris Dataset', fontsize=16)
plt.tight_layout()
plt.show()

## Model Evaluation

In [ ]:
# Classification report
print("Classification Report:")
print(classification_report(y_test, y_pred_sklearn, target_names=target_names))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_sklearn)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=target_names,
            yticklabels=target_names)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix - Decision Stump')
plt.tight_layout()
plt.show()

## Decision Boundary Visualization

Visualize the decision boundary created by the decision stump (using only 2 features).

In [ ]:
# Use only 2 features for visualization
X_2d = X[:, :2]  # Use first 2 features
feature_names_2d = feature_names[:2]

X_train_2d, X_test_2d, y_train_2d, y_test_2d = train_test_split(
    X_2d, y, test_size=0.3, random_state=42, stratify=y
)

# Train stump on 2D data
stump_2d = DecisionTreeClassifier(max_depth=1, random_state=42)
stump_2d.fit(X_train_2d, y_train_2d)

# Create mesh grid
h = 0.02
x_min, x_max = X_2d[:, 0].min() - 1, X_2d[:, 0].max() + 1
y_min, y_max = X_2d[:, 1].min() - 1, X_2d[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))

# Predict on mesh grid
Z = stump_2d.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

# Plot decision boundary
plt.figure(figsize=(12, 8))
plt.contourf(xx, yy, Z, alpha=0.3, cmap='viridis')
scatter = plt.scatter(X_2d[:, 0], X_2d[:, 1], c=y, cmap='viridis', edgecolors='k')
plt.xlabel(feature_names_2d[0])
plt.ylabel(feature_names_2d[1])
plt.title('Decision Stump Decision Boundary (2D)')
plt.colorbar(scatter, label='Class')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Comparison: Decision Stump vs Full Decision Tree

In [ ]:
# Compare with full decision tree
full_tree = DecisionTreeClassifier(random_state=42)
full_tree.fit(X_train, y_train)
y_pred_full = full_tree.predict(X_test)
accuracy_full = accuracy_score(y_test, y_pred_full)

print("Comparison:")
print(f"Decision Stump Accuracy: {accuracy_sklearn:.4f}")
print(f"Full Decision Tree Accuracy: {accuracy_full:.4f}")
print(f"\nDecision Stump Depth: {stump_sklearn.get_depth()}")
print(f"Full Decision Tree Depth: {full_tree.get_depth()}")
print(f"\nDecision Stump Leaves: {stump_sklearn.get_n_leaves()}")
print(f"Full Decision Tree Leaves: {full_tree.get_n_leaves()}")

## Decision Stump in AdaBoost

Decision stumps are commonly used as weak learners in AdaBoost.

In [ ]:
# AdaBoost with decision stumps
adaboost = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1),  # Decision stumps
    n_estimators=50,
    learning_rate=1.0,
    random_state=42
)

adaboost.fit(X_train, y_train)
y_pred_ada = adaboost.predict(X_test)
accuracy_ada = accuracy_score(y_test, y_pred_ada)

print(f"AdaBoost with Decision Stumps Accuracy: {accuracy_ada:.4f}")
print(f"Number of estimators: {adaboost.n_estimators}")

In [ ]:
# Compare all models
print("Model Comparison:")
print("=" * 50)
print(f"Decision Stump:        {accuracy_sklearn:.4f}")
print(f"Full Decision Tree:    {accuracy_full:.4f}")
print(f"AdaBoost (50 stumps):  {accuracy_ada:.4f}")

## Dataset 2: Synthetic Binary Classification

In [ ]:
# Create synthetic binary classification dataset
X_syn, y_syn = make_classification(
    n_samples=1000,
    n_features=5,
    n_informative=3,
    n_redundant=1,
    n_classes=2,
    n_clusters_per_class=1,
    random_state=42
)

print(f"Synthetic dataset shape: {X_syn.shape}")
print(f"Class distribution: {np.bincount(y_syn)}")

In [ ]:
# Split and train
X_train_syn, X_test_syn, y_train_syn, y_test_syn = train_test_split(
    X_syn, y_syn, test_size=0.3, random_state=42, stratify=y_syn
)

# Train decision stump
stump_syn = DecisionTreeClassifier(max_depth=1, random_state=42)
stump_syn.fit(X_train_syn, y_train_syn)
y_pred_syn = stump_syn.predict(X_test_syn)
accuracy_syn = accuracy_score(y_test_syn, y_pred_syn)

print(f"Decision Stump Accuracy (Binary): {accuracy_syn:.4f}")

In [ ]:
# Feature importance
feature_importance = stump_syn.feature_importances_

importance_df = pd.DataFrame({
    'Feature': [f'Feature_{i}' for i in range(X_syn.shape[1])],
    'Importance': feature_importance
}).sort_values('Importance', ascending=False)

print("Feature Importance:")
print(importance_df)

# Plot
plt.figure(figsize=(10, 6))
sns.barplot(data=importance_df, x='Importance', y='Feature')
plt.title('Feature Importance - Decision Stump')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

## Cross-Validation

In [ ]:
# Cross-validation for decision stump
cv_scores = cross_val_score(stump_sklearn, X_train, y_train, cv=5)

print(f"Cross-validation scores: {cv_scores}")
print(f"Mean CV accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")

## Predict on New Data

In [ ]:
# Function to predict with decision stump
def predict_with_stump(sample):
    prediction = stump_sklearn.predict([sample])[0]
    probabilities = stump_sklearn.predict_proba([sample])[0]
    
    class_name = target_names[prediction]
    return class_name, probabilities

# Test with new samples
new_samples = [
    [5.1, 3.5, 1.4, 0.2],  # Likely setosa
    [6.3, 2.8, 5.1, 1.5],  # Likely versicolor
    [6.4, 3.2, 5.3, 2.3]   # Likely virginica
]

for sample in new_samples:
    class_name, probs = predict_with_stump(sample)
    print(f"\nSample: {sample}")
    print(f"Predicted: {class_name}")
    print("Probabilities:")
    for name, prob in zip(target_names, probs):
        print(f"  {name}: {prob:.4f}")

## Summary

### Key Takeaways:

1. **Single Split**: Makes only one decision based on one feature
2. **Weak Learner**: Often used as weak learner in ensemble methods
3. **Highly Interpretable**: Very easy to understand and explain
4. **Fast Training**: Extremely fast due to simplicity
5. **Baseline Model**: Useful as a baseline for comparison

### Advantages:
- Extremely simple and interpretable
- Very fast to train and predict
- Low computational cost
- No overfitting (too simple to overfit)
- Excellent as weak learner in boosting
- Good baseline model
- Provides feature importance

### Limitations:
- Very limited predictive power
- Can only capture simple patterns
- Underfits most complex datasets
- Only uses one feature
- Poor performance on non-linearly separable data
- Not suitable for complex classification tasks alone

### Use Cases:

- **Weak Learner**: In AdaBoost and other boosting algorithms
- **Baseline**: As a simple baseline to compare against more complex models
- **Feature Selection**: To identify the most important single feature
- **Quick Prototyping**: When you need a very fast initial model
- **Interpretability**: When you need the most interpretable model possible

### Decision Stump vs Full Decision Tree:

- **Decision Stump**: max_depth=1, single split, weak learner, fast
- **Full Tree**: unlimited depth, multiple splits, strong learner, can overfit